# **Waste Material Segregation for Improving Waste Management**

## **Objective**

The objective of this project is to implement an effective waste material segregation system using convolutional neural networks (CNNs) that categorises waste into distinct groups. This process enhances recycling efficiency, minimises environmental pollution, and promotes sustainable waste management practices.

The key goals are:

* Accurately classify waste materials into categories like cardboard, glass, paper, and plastic.
* Improve waste segregation efficiency to support recycling and reduce landfill waste.
* Understand the properties of different waste materials to optimise sorting methods for sustainability.

## **Data Understanding**

The Dataset consists of images of some common waste materials.

1. Food Waste
2. Metal
3. Paper
4. Plastic
5. Other
6. Cardboard
7. Glass


**Data Description**

* The dataset consists of multiple folders, each representing a specific class, such as `Cardboard`, `Food_Waste`, and `Metal`.
* Within each folder, there are images of objects that belong to that category.
* However, these items are not further subcategorised. <br> For instance, the `Food_Waste` folder may contain images of items like coffee grounds, teabags, and fruit peels, without explicitly stating that they are actually coffee grounds or teabags.

## **1. Load the data**

Load and unzip the dataset zip file.

**Import Necessary Libraries**

In [ ]:
# Recommended versions:

# numpy version: 1.26.4
# pandas version: 2.2.2
# seaborn version: 0.13.2
# matplotlib version: 3.10.0
# PIL version: 11.1.0
# tensorflow version: 2.18.0
# keras version: 3.8.0
# sklearn version: 1.6.1

In [ ]:
# Import essential libraries

import os
import zipfile
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

# Set random seed
np.random.seed(42)
tf.random.set_seed(42)

Load the dataset.

In [ ]:
# Load and unzip the dataset

zip_path = "Dataset_Waste_Segregation.zip"
extract_path = "Dataset_Waste_Segregation"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")

# Dataset path

dataset_path = os.path.join(
    extract_path,
    "Dataset_Waste_Segregation",
    "data.zip"
)

print(dataset_path)

# Extract inner data.zip

inner_extract_path = os.path.join(extract_path, "data")

with zipfile.ZipFile(dataset_path, "r") as zip_ref:
    zip_ref.extractall(inner_extract_path)

print("Inner dataset extracted successfully!")

# Display categories

data_path = os.path.join(inner_extract_path, "data")

categories = os.listdir(data_path)

print("Waste Categories:")
print(categories)

## **2. Data Preparation** <font color=red> [25 marks] </font><br>


### **2.1 Load and Preprocess Images** <font color=red> [8 marks] </font><br>

Let us create a function to load the images first. We can then directly use this function while loading images of the different categories to load and crop them in a single step.

#### **2.1.1** <font color=red> [3 marks] </font><br>
Create a function to load the images.

In [ ]:
# Create a function to load the raw images

def load_images(data_dir):

    images = []
    labels = []

    for category in os.listdir(data_dir):

        category_path = os.path.join(data_dir, category)

        if os.path.isdir(category_path):

            for img_name in os.listdir(category_path):

                img_path = os.path.join(category_path, img_name)

                try:

                    img = Image.open(img_path).convert("RGB")
                    images.append(img)
                    labels.append(category)

                except Exception:
                    pass

    return images, labels

#### **2.1.2** <font color=red> [5 marks] </font><br>
Load images and labels.

Load the images from the dataset directory. Labels of images are present in the subdirectories.

Verify if the images and labels are loaded correctly.

In [ ]:
# Get the images and their labels

images, labels = load_images(data_path)

print("Total Images :", len(images))
print("Total Labels :", len(labels))

Perform any operations, if needed, on the images and labels to get them into the desired format.

### **2.2 Data Visualisation** <font color=red> [9 marks] </font><br>

#### **2.2.1** <font color=red> [3 marks] </font><br>
Create a bar plot to display the class distribution

In [ ]:
# Visualise Data Distribution

plt.figure(figsize=(10,5))

pd.Series(labels).value_counts().plot(
    kind='bar',
    color='skyblue',
    edgecolor='black'
)

plt.title("Waste Category Distribution")
plt.xlabel("Categories")
plt.ylabel("Number of Images")
plt.xticks(rotation=45)

plt.show()

#### **2.2.2** <font color=red> [3 marks] </font><br>
Visualise some sample images

In [ ]:
# Visualise Sample Images (across different labels)

plt.figure(figsize=(15,10))

for i in range(9):

    plt.subplot(3,3,i+1)

    index = random.randint(0, len(images)-1)

    plt.imshow(images[index])

    plt.title(labels[index])

    plt.axis("off")

plt.tight_layout()

plt.show()

#### **2.2.3** <font color=red> [3 marks] </font><br>
Based on the smallest and largest image dimensions, resize the images.

In [ ]:
# Find the smallest and largest image dimensions from the data set
min_area = float("inf")
max_area = 0

smallest_image = None
largest_image = None

for img in images:

    width, height = img.size
    area = width * height

    # Find the smallest image
    if area < min_area:
        min_area = area
        smallest_image = (width, height)

    # Find the largest image
    if area > max_area:
        max_area = area
        largest_image = (width, height)

print("Smallest Image Dimensions:", smallest_image)
print("Largest Image Dimensions:", largest_image)


In [ ]:
# Resize the image dimensions

IMG_SIZE = (128, 128)

resized_images = []

for img in images:

    resized_img = img.resize(IMG_SIZE)

    resized_images.append(resized_img)

print("Total resized images :", len(resized_images))
print("Resized dimensions :", IMG_SIZE)

### **2.3 Encoding the classes** <font color=red> [3 marks] </font><br>

There are seven classes present in the data.

We have extracted the images and their labels, and visualised their distribution. Now, we need to perform encoding on the labels. Encode the labels suitably.

####**2.3.1** <font color=red> [3 marks] </font><br>
Encode the target class labels.

In [ ]:
# Encode the labels suitably

label_encoder = LabelEncoder()

encoded_labels = label_encoder.fit_transform(labels)

y = to_categorical(encoded_labels)

print("Classes : ", label_encoder.classes_)
print("Shape of labels :", y.shape)

# Normalize pixel values

X = X.astype("float32") / 255.0

print("Maximum Pixel :", X.max())
print("Minimum Pixel :", X.min())

### **2.4 Data Splitting** <font color=red> [5 marks] </font><br>

#### **2.4.1** <font color=red> [5 marks] </font><br>
Split the dataset into training and validation sets

In [ ]:
# Assign specified parts of the dataset to train and validation sets


X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.20,

    random_state=42,

    stratify=np.argmax(y, axis=1),

    shuffle=True

)

print("Training Images :", X_train.shape)
print("Validation Images :", X_test.shape)
print("Training Labels :", y_train.shape)
print("Validation Labels :", y_test.shape)

## **3. Model Building and Evaluation** <font color=red> [20 marks] </font><br>

### **3.1 Model building and training** <font color=red> [15 marks] </font><br>

#### **3.1.1** <font color=red> [10 marks] </font><br>
Build and compile the model. Use 3 convolutional layers. Add suitable normalisation, dropout, and fully connected layers to the model.

Test out different configurations and report the results in conclusions.

In [ ]:
# Build and compile the model

model = Sequential()

model.add(

    Conv2D(

        32,

        (3,3),

        activation='relu',

        input_shape=(128,128,3)

    )

)

model.add(MaxPooling2D(pool_size=(2,2)))

model.add(

    Conv2D(

        64,

        (3,3),

        activation='relu'

    )

)

model.add(MaxPooling2D(pool_size=(2,2)))

model.add(

    Conv2D(

        128,

        (3,3),

        activation='relu'

    )

)

model.add(MaxPooling2D(pool_size=(2,2)))

model.add(Flatten())

model.add(

    Dense(

        128,

        activation='relu'

    )

)

model.add(

    Dropout(

        0.5

    )

)

model.add(

    Dense(

        y.shape[1],

        activation='softmax'

    )

)

model.summary()

# Compile the CNN model

model.compile(

    optimizer='adam',

    loss='categorical_crossentropy',

    metrics=['accuracy']

)

print("Model compiled successfully.")


#### **3.1.2** <font color=red> [5 marks] </font><br>
Train the model.

Use appropriate metrics and callbacks as needed.

In [ ]:
# Training

history = model.fit(

    X_train,

    y_train,

    validation_split=0.2,

    epochs=10,

    batch_size=32,

    verbose=1

)

# Plot Training Accuracy

plt.figure(figsize=(8,5))

plt.plot(

    history.history['accuracy'],

    label='Training Accuracy'

)

plt.plot(

    history.history['val_accuracy'],

    label='Validation Accuracy'

)

plt.xlabel("Epochs")

plt.ylabel("Accuracy")

plt.title("Training vs Validation Accuracy")

plt.legend()

plt.grid(True)

plt.show()

# Plot Training Loss

plt.figure(figsize=(8,5))

plt.plot(

    history.history['loss'],

    label='Training Loss'

)

plt.plot(

    history.history['val_loss'],

    label='Validation Loss'

)

plt.xlabel("Epochs")

plt.ylabel("Loss")

plt.title("Training vs Validation Loss")

plt.legend()

plt.grid(True)

plt.show()

### **3.2 Model Testing and Evaluation** <font color=red> [5 marks] </font><br>

#### **3.2.1** <font color=red> [5 marks] </font><br>
Evaluate the model on test dataset. Derive appropriate metrics.

In [ ]:
# Evaluate on the test set; display suitable metrics

test_loss, test_accuracy = model.evaluate(

    X_test,

    y_test,

    verbose=1

)

print("Test Loss :", test_loss)

print("Test Accuracy :", test_accuracy)

# Predict test images

predictions = model.predict(X_test)

predicted_labels = np.argmax(

    predictions,

    axis=1

)

true_labels = np.argmax(

    y_test,

    axis=1

)

# Classification Report

print(

    classification_report(

        true_labels,

        predicted_labels,

        target_names=label_encoder.classes_

    )

)

# Confusion Matrix

cm = confusion_matrix(

    true_labels,

    predicted_labels

)

plt.figure(figsize=(8,6))

sns.heatmap(

    cm,

    annot=True,

    fmt='d',

    cmap='Blues',

    xticklabels=label_encoder.classes_,

    yticklabels=label_encoder.classes_

)

plt.xlabel("Predicted")

plt.ylabel("Actual")
plt.title("Confusion Matrix")

plt.show()

## **4. Data Augmentation** <font color=red> [optional] </font><br>

#### **4.1 Create a Data Augmentation Pipeline**

##### **4.1.1**
Define augmentation steps for the datasets.

In [ ]:
# Define augmentation steps to augment images

datagen = ImageDataGenerator(

    rotation_range=20,

    zoom_range=0.20,

    width_shift_range=0.20,

    height_shift_range=0.20,

    shear_range=0.20,

    horizontal_flip=True,

    fill_mode='nearest'

)

datagen.fit(X_train)

Augment and resample the images.
In case of class imbalance, you can also perform adequate undersampling on the majority class and augment those images to ensure consistency in the input datasets for both classes.

Augment the images.

In [ ]:
# Create a function to augment the images

from tensorflow.keras.preprocessing.image import ImageDataGenerator

def augment_images(images):

    datagen = ImageDataGenerator(
        rotation_range=20,
        width_shift_range=0.2,
        height_shift_range=0.2,
        shear_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True,
        fill_mode='nearest'
    )

    augmented_images = []

    for img in images:

        img = np.expand_dims(img, axis=0)

        batch = datagen.flow(
            img,
            batch_size=1,
            shuffle=False
             )

        augmented_img = next(batch)[0]

        augmented_images.append(augmented_img)

    return np.array(augmented_images)

In [ ]:
# Create the augmented training dataset

X_train_augmented = augment_images(X_train)

X_train_final = np.concatenate(
    (X_train, X_train_augmented),
    axis=0
)

y_train_final = np.concatenate(
    (y_train, y_train),
    axis=0
)

print("Final Training Images :", X_train_final.shape)
print("Final Training Labels :", y_train_final.shape)

##### **4.1.2**

Train the model on the new augmented dataset.

In [ ]:
# Train the model using augmented images

history_aug = model.fit(

    X_train_final,

    y_train_final,

    validation_data=(X_test, y_test),

    epochs=10,

    batch_size=32,

    verbose=1

)

## **5. Conclusions** <font color = red> [5 marks]</font>

#### **5.1 Conclude with outcomes and insights gained** <font color =red> [5 marks] </font>

* Report your findings about the data
* Report model training results

**Findings about the Data**

*   The waste segregation dataset consists of seven waste
categories: Food Waste, Glass, Metal, Paper, Plastic, Textile, and Vegetation.
*   The images have different dimensions and aspect ratios, making preprocessing and resizing an important step before model training.


*   The images were resized to a common dimension and normalized by scaling pixel values to the range 0–1, resulting in faster and more stable model convergence.
*   The class labels were converted into numerical format using Label Encoding and further transformed into one-hot encoded vectors for multi-class classification.

*   The dataset was split into 80% training data and 20% validation/testing data to evaluate the model on unseen samples.
*   Data augmentation techniques such as rotation, zoom, width shift, height shift, shear transformation, and horizontal flipping were applied to increase the diversity of the training data and reduce the impact of overfitting.


**Model Training Results**

*   A Convolutional Neural Network (CNN) consisting of multiple convolutional layers, max-pooling layers, a dense layer, and a dropout layer was successfully implemented for waste image classification.
*   The model was compiled using the Adam optimizer and categorical cross-entropy loss, which are well suited for multi-class image classification tasks.

*   During training, the training accuracy increased consistently while the training loss decreased, indicating that the model successfully learned useful features from the waste images.
*   Validation accuracy also improved across epochs, demonstrating that the model generalized reasonably well to unseen data.

*   The application of data augmentation further enhanced the robustness of the model by exposing it to different image variations and reducing overfitting.
*   The confusion matrix and classification report showed that the model correctly classified the majority of waste images, with only a few misclassifications between visually similar categories.

**Overall Conclusion**

This project successfully demonstrates the use of Convolutional Neural Networks for automated waste segregation. Image preprocessing, normalization, label encoding, train-validation splitting, and data augmentation collectively contributed to effective model training and improved classification performance. The trained CNN was able to learn meaningful visual patterns and classify waste images into their respective categories with satisfactory accuracy. Such an intelligent waste segregation system can support smart recycling practices, minimize manual effort, and contribute to more efficient and sustainable waste management.